In [ ]:
## ✅ Final Notebook Code

# %% [markdown]
# # Transfer Learning for Manufacturing Defect Detection with ResNet18
#
# ## 🎯 Goal
#
# This notebook teaches transfer learning end-to-end by fine-tuning a
# pre-trained ResNet18 for **binary image classification** in a manufacturing
# defect-detection setting.
#
# ## 📁 Expected Input Dataset Structure
#
# The notebook expects RGB images organized as:
#
# ```text
# data/defects/
# ├── train/
# │   ├── normal/
# │   └── defective/
# └── val/
#     ├── normal/
#     └── defective/
# ```
#
# The target dataset is assumed to be:
#
# - Small: approximately 400 training images and 100 validation images
# - Binary: `normal` vs. `defective`
# - Imbalanced: approximately 80% normal / 20% defective
#
# ## 📦 Outputs
#
# This notebook produces four trained model artifacts:
#
# 1. `artifacts/resnet18_full_freeze.pth`
# 2. `artifacts/resnet18_layer4_unfreeze.pth`
# 3. `artifacts/resnet18_peft_adapter.pth`
# 4. `artifacts/resnet18_from_scratch.pth`
#
# Each variant also saves a per-epoch training log as JSON.
#
# ## ⚠️ CPU-Only Teaching Trade-off
#
# This notebook prioritizes **clarity and memory efficiency** over speed.
# It is designed to run on CPU. To keep the 4-way comparison practical,
# the default configuration uses a small number of epochs and optionally
# caps the dataset size. Increase these values for real experiments.

# %% [markdown]
# ---
# # Section 0 — Setup & Framing
#
# In this section we import all dependencies, configure the device, define
# paths, and prepare helper utilities.
#
# We also attempt to import the `peft` library. If it is unavailable or fails
# on torchvision ResNet convolution layers, the notebook falls back to a small
# manual adapter implementation.

# %%
from __future__ import annotations

import copy
import json
import math
import os
import random
import shutil
import tarfile
import urllib.request
from collections import defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from torchvision.models import ResNet18_Weights

# PEFT is optional. We attempt to import it, but keep a manual fallback.
try:
    from peft import LoraConfig, get_peft_model

    PEFT_AVAILABLE = True
except Exception as exc:
    print(f"PEFT import failed; manual adapter fallback will be used. Error: {exc}")
    PEFT_AVAILABLE = False

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device detection with automatic CPU fallback
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Paths
DATA_ROOT = Path("data/defects")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# CPU-friendly defaults
BATCH_SIZE = 16
NUM_WORKERS = 0
NUM_EPOCHS_TRANSFER = 2
NUM_EPOCHS_BASELINE = 2

# Set to None to use the full dataset.
# Keeping these small makes the 4-way comparison practical on CPU.
MAX_TRAIN_SAMPLES = 160
MAX_VAL_SAMPLES = 80

# ImageNet normalization used by pre-trained ResNet18
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# %% [markdown]
# ---
# # Section 1 — The Pre-trained Model
#
# ## What does "pre-trained" mean?
#
# A pre-trained model has already learned useful parameters on a large source
# dataset before we adapt it to our target task.
#
# For ResNet18, the commonly used pre-trained weights come from **ImageNet**:
#
# - Source dataset: ImageNet
# - Source task: 1000-class object classification
# - Source domain: natural images
#
# The weights store learned convolutional filters, batch-normalization
# statistics, and classifier parameters. Early filters often detect generic
# visual patterns such as edges, corners, textures, and color blobs. These
# features can generalize surprisingly well to other visual domains.
#
# However, manufacturing defect images may differ from ImageNet photos, so
# we must decide how much of the network to reuse and how much to fine-tune.

# %%
pretrained_model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
print(pretrained_model)

# %%
print("Named children:")
for name, child in pretrained_model.named_children():
    print(f"{name}: {child.__class__.__name__}")

# %%
print("First 20 named parameters:")
for idx, (name, param) in enumerate(pretrained_model.named_parameters()):
    print(f"{idx:02d} | {name:40s} | shape={tuple(param.shape)}")
    if idx >= 19:
        break

# %% [markdown]
# ---
# # Section 2 — Source Task vs. Target Task
#
# | Aspect | Source Task | Target Task |
# |---|---|---|
# | Dataset | ImageNet | Manufacturing defect dataset |
# | Classes | 1000 object classes | 2 classes: normal, defective |
# | Domain | Natural images | Industrial / manufacturing images |
# | Typical scale | Millions of images | Very small, around 500 total images |
# | Class balance | Relatively broad | Highly imbalanced, about 80/20 |
#
# ## Checkpoint: Domain Similarity
#
# Before changing the model, ask:
#
# > Are the target images visually similar to ImageNet natural images, or do
# > they represent a domain shift?
#
# Manufacturing images may share low-level visual features with ImageNet,
# such as edges, textures, and lighting patterns. However, defects can be
# subtle and domain-specific. This suggests:
#
# - Early ResNet features may transfer well.
# - Later task-specific layers may need adaptation.
# - With a very small and imbalanced dataset, full fine-tuning may overfit.
#
# Therefore, Section 5 compares:
#
# 1. Full feature freeze
# 2. Controlled `layer4` unfreezing
# 3. Parameter-efficient adapter tuning
# 4. From-scratch baseline

# %% [markdown]
# ## Automated Dataset Handling
#
# If the expected input folders are empty, this notebook downloads the real
# photographic Hymenoptera dataset from PyTorch tutorials and redistributes
# the images into the required `normal` / `defective` folders.
#
# ⚠️ No dummy arrays, synthetic NumPy images, or solid-color placeholders are
# generated. The fallback uses real images only.
#
# The labels in this fallback are only for demonstration: images are assigned
# into `normal` and `defective` folders to simulate an 80/20 imbalance.

# %%
REQUIRED_SUBDIRS = [
    DATA_ROOT / "train" / "normal",
    DATA_ROOT / "train" / "defective",
    DATA_ROOT / "val" / "normal",
    DATA_ROOT / "val" / "defective",
]


def ensure_directories() -> None:
    """Create expected dataset directories if they do not exist."""
    for directory in REQUIRED_SUBDIRS:
        directory.mkdir(parents=True, exist_ok=True)


def directory_has_images(directory: Path) -> bool:
    """Return True if a directory contains at least one supported image."""
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return any(path.suffix.lower() in extensions for path in directory.rglob("*"))


def required_folders_are_empty() -> bool:
    """Check whether all required class folders are empty."""
    ensure_directories()
    return not any(directory_has_images(directory) for directory in REQUIRED_SUBDIRS)


def download_hymenoptera(download_root: Path) -> Path:
    """Download and extract the Hymenoptera dataset."""
    download_root.mkdir(parents=True, exist_ok=True)
    url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
    zip_path = download_root / "hymenoptera_data.zip"
    extract_dir = download_root / "hymenoptera_data"

    if not extract_dir.exists():
        print("Downloading Hymenoptera dataset...")
        urllib.request.urlretrieve(url, zip_path)

        print("Extracting Hymenoptera dataset...")
        shutil.unpack_archive(str(zip_path), str(download_root))

    return extract_dir


def collect_real_images(source_root: Path) -> List[Path]:
    """Collect real image paths from the downloaded dataset."""
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    image_paths = [
        path
        for path in source_root.rglob("*")
        if path.suffix.lower() in extensions and path.is_file()
    ]
    random.shuffle(image_paths)
    return image_paths


def copy_split(
    image_paths: List[Path],
    destination_root: Path,
    train_fraction: float = 0.8,
    normal_fraction: float = 0.8,
) -> None:
    """Redistribute real images into train/val and normal/defective folders."""
    ensure_directories()

    # Clear existing files only because this function is called when folders
    # are empty or created for fallback demo data.
    for directory in REQUIRED_SUBDIRS:
        for file_path in directory.glob("*"):
            if file_path.is_file():
                file_path.unlink()

    num_images = len(image_paths)
    train_count = int(train_fraction * num_images)

    split_map = {
        "train": image_paths[:train_count],
        "val": image_paths[train_count:],
    }

    for split_name, split_images in split_map.items():
        num_normal = int(normal_fraction * len(split_images))

        for idx, src_path in enumerate(split_images):
            class_name = "normal" if idx < num_normal else "defective"
            dst_dir = destination_root / split_name / class_name
            dst_name = f"{src_path.stem}_{idx}{src_path.suffix}"
            shutil.copy2(src_path, dst_dir / dst_name)

    print("Fallback dataset created with real photographic images.")
    print(f"Dataset root: {destination_root.resolve()}")


ensure_directories()

if required_folders_are_empty():
    hymenoptera_root = download_hymenoptera(Path("data/external"))
    real_images = collect_real_images(hymenoptera_root)
    copy_split(real_images, DATA_ROOT)
else:
    print("Using existing dataset folders.")

# %%
def count_images_by_folder(root: Path) -> pd.DataFrame:
    """Count images in each split/class folder."""
    records = []
    for split in ["train", "val"]:
        for class_name in ["normal", "defective"]:
            folder = root / split / class_name
            count = sum(1 for _ in folder.glob("*") if _.is_file())
            records.append(
                {
                    "split": split,
                    "class": class_name,
                    "count": count,
                }
            )
    return pd.DataFrame(records)


dataset_counts = count_images_by_folder(DATA_ROOT)
dataset_counts

# %% [markdown]
# ---
# # Section 3 — Feature Reuse: Visualizing What Transfers
#
# Transfer learning works because many visual features learned on a large
# source dataset can be reused.
#
# In convolutional networks:
#
# - **Early layers** tend to learn generic visual features:
#   - edges
#   - corners
#   - simple textures
#   - color contrasts
#
# - **Late layers** tend to learn task-specific combinations:
#   - object parts
#   - semantic shapes
#   - class-specific patterns
#
# Here we hook into an early layer and a late layer of ResNet18 and visualize
# activation maps for a target-domain image.

# %%
visualization_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

visualization_dataset = datasets.ImageFolder(
    root=DATA_ROOT / "val",
    transform=visualization_transform,
)

print(f"Visualization dataset classes: {visualization_dataset.classes}")

# %%
def denormalize_image(tensor: torch.Tensor) -> np.ndarray:
    """Convert normalized CHW tensor to displayable HWC image."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    image = tensor.cpu() * std + mean
    image = image.clamp(0, 1)
    return image.permute(1, 2, 0).numpy()


def visualize_activations(
    model: nn.Module,
    image_tensor: torch.Tensor,
    device: torch.device,
) -> None:
    """Visualize mean activation maps from early and late ResNet layers."""
    model = model.to(device)
    model.eval()

    activations = {}

    def make_hook(name: str):
        def hook(_module, _input, output):
            activations[name] = output.detach().cpu()

        return hook

    hooks = [
        model.conv1.register_forward_hook(make_hook("early_conv1")),
        model.layer4[-1].conv2.register_forward_hook(make_hook("late_layer4")),
    ]

    with torch.no_grad():
        _ = model(image_tensor.unsqueeze(0).to(device))

    for hook in hooks:
        hook.remove()

    original = denormalize_image(image_tensor)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(original)
    axes[0].set_title("Input target image")
    axes[0].axis("off")

    for axis, name in zip(axes[1:], ["early_conv1", "late_layer4"]):
        activation = activations[name][0]
        activation_map = activation.mean(dim=0)
        axis.imshow(activation_map, cmap="viridis")
        axis.set_title(name)
        axis.axis("off")

    plt.tight_layout()
    plt.show()


sample_image, sample_label = visualization_dataset[0]
print(f"Sample label: {visualization_dataset.classes[sample_label]}")

feature_model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
visualize_activations(feature_model, sample_image, DEVICE)

# %% [markdown]
# ---
# # Section 4 — Network Modification
#
# ResNet18 was originally trained to classify ImageNet images into 1000
# classes. Our target task has only 2 classes:
#
# - `normal`
# - `defective`
#
# Therefore, we replace the final fully connected classifier:
#
# ```python
# model.fc = nn.Linear(in_features, num_target_classes)
# ```
#
# This is the central network modification in many transfer-learning workflows:
#
# - Reuse the convolutional feature extractor.
# - Replace the task-specific classification head.

# %%
NUM_TARGET_CLASSES = 2


def create_resnet18(
    pretrained: bool = True,
    num_classes: int = NUM_TARGET_CLASSES,
) -> nn.Module:
    """Create a ResNet18 model and replace the final classifier."""
    weights = ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


modified_model = create_resnet18(pretrained=True, num_classes=NUM_TARGET_CLASSES)
print(modified_model.fc)

# %% [markdown]
# ---
# # Section 5 — Freeze vs. Fine-tune vs. PEFT
#
# This section defines the three transfer-learning variants.
#
# ## Variant 1: Full Freeze
#
# Freeze all convolutional layers and train only the new classification head.
#
# This is the default strategy for a very small dataset because it reduces
# overfitting risk.
#
# ## Variant 2: Unfreeze `layer4`
#
# Freeze most of the model but fine-tune the last residual block group.
#
# This can help when the target domain differs from ImageNet enough that the
# most task-specific features need adaptation.
#
# With class imbalance, this must be done carefully because the model may
# overfit to majority-class patterns.
#
# ## Variant 3: PEFT / Adapter Fine-tuning
#
# Parameter-efficient fine-tuning trains only a small number of extra
# parameters while keeping the main backbone mostly frozen.
#
# The notebook first attempts to use `peft`. If that fails or is unavailable,
# it falls back to a minimal manual convolutional adapter.

# %%
def freeze_all_parameters(model: nn.Module) -> None:
    """Freeze every parameter in the model."""
    for parameter in model.parameters():
        parameter.requires_grad = False


def unfreeze_module(module: nn.Module) -> None:
    """Unfreeze every parameter inside a module."""
    for parameter in module.parameters():
        parameter.requires_grad = True


def count_trainable_parameters(model: nn.Module) -> Tuple[int, int]:
    """Return trainable and total parameter counts."""
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
    return trainable, total


class ConvAdapter(nn.Module):
    """A small residual adapter around an existing Conv2d layer.

    The original convolution is frozen. The adapter learns a small residual
    correction using two 1x1 convolutions.
    """

    def __init__(
        self,
        conv: nn.Conv2d,
        rank: int = 4,
        alpha: float = 1.0,
    ) -> None:
        super().__init__()
        self.conv = conv

        for parameter in self.conv.parameters():
            parameter.requires_grad = False

        in_channels = conv.in_channels
        out_channels = conv.out_channels
        self.scale = alpha / rank

        self.down = nn.Conv2d(
            in_channels,
            rank,
            kernel_size=1,
            stride=conv.stride,
            bias=False,
        )
        self.up = nn.Conv2d(
            rank,
            out_channels,
            kernel_size=1,
            stride=1,
            bias=False,
        )

        nn.init.kaiming_normal_(self.down.weight, nonlinearity="relu")
        nn.init.zeros_(self.up.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply frozen convolution plus trainable adapter residual."""
        return self.conv(x) + self.scale * self.up(self.down(x))


def inject_manual_adapters_into_layer4(
    model: nn.Module,
    rank: int = 4,
    alpha: float = 1.0,
) -> nn.Module:
    """Inject small adapters into Conv2d layers inside ResNet layer4."""
    for block in model.layer4:
        block.conv1 = ConvAdapter(block.conv1, rank=rank, alpha=alpha)
        block.conv2 = ConvAdapter(block.conv2, rank=rank, alpha=alpha)
    return model


def apply_peft_or_manual_adapter(model: nn.Module) -> nn.Module:
    """Apply PEFT LoRA if possible, otherwise use manual adapters."""
    freeze_all_parameters(model)

    # Always keep the final classification head trainable.
    unfreeze_module(model.fc)

    if PEFT_AVAILABLE:
        try:
            # PEFT API can vary across versions and may not support every
            # Conv2d use case in torchvision models. We catch failures and
            # use the manual adapter fallback below.
            peft_config = LoraConfig(
                r=4,
                lora_alpha=8,
                target_modules=["conv1", "conv2"],
                lora_dropout=0.05,
                bias="none",
                modules_to_save=["fc"],
            )
            model = get_peft_model(model, peft_config)
            print("Using PEFT LoRA implementation.")
            return model
        except Exception as exc:
            print("PEFT failed on this model/runtime.")
            print(f"Falling back to manual adapters. Error: {exc}")

    model = inject_manual_adapters_into_layer4(model, rank=4, alpha=1.0)
    print("Using manual ConvAdapter fallback.")
    return model


def train_full_freeze(
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    class_names: List[str],
    device: torch.device,
    num_epochs: int = NUM_EPOCHS_TRANSFER,
) -> Tuple[nn.Module, List[Dict[str, float]]]:
    """Train only the final FC head of a fresh pre-trained ResNet18."""
    model = create_resnet18(pretrained=True, num_classes=len(class_names))
    freeze_all_parameters(model)
    unfreeze_module(model.fc)

    trainable, total = count_trainable_parameters(model)
    print(f"Full-freeze trainable params: {trainable:,} / {total:,}")

    optimizer = optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-4)

    model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        num_epochs=num_epochs,
        artifact_path=ARTIFACT_DIR / "resnet18_full_freeze.pth",
        log_path=ARTIFACT_DIR / "resnet18_full_freeze_log.json",
        freeze_batchnorm=True,
    )
    return model, history


def train_unfreeze_layer4(
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    class_names: List[str],
    device: torch.device,
    num_epochs: int = NUM_EPOCHS_TRANSFER,
) -> Tuple[nn.Module, List[Dict[str, float]]]:
    """Fine-tune layer4 and the FC head of a fresh pre-trained ResNet18."""
    model = create_resnet18(pretrained=True, num_classes=len(class_names))
    freeze_all_parameters(model)
    unfreeze_module(model.layer4)
    unfreeze_module(model.fc)

    trainable, total = count_trainable_parameters(model)
    print(f"Layer4-unfreeze trainable params: {trainable:,} / {total:,}")

    optimizer = optim.AdamW(
        filter(lambda parameter: parameter.requires_grad, model.parameters()),
        lr=3e-4,
        weight_decay=1e-4,
    )

    model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        num_epochs=num_epochs,
        artifact_path=ARTIFACT_DIR / "resnet18_layer4_unfreeze.pth",
        log_path=ARTIFACT_DIR / "resnet18_layer4_unfreeze_log.json",
        freeze_batchnorm=True,
    )
    return model, history


def train_peft(
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    class_names: List[str],
    device: torch.device,
    num_epochs: int = NUM_EPOCHS_TRANSFER,
) -> Tuple[nn.Module, List[Dict[str, float]]]:
    """Train a PEFT/adapter variant of a fresh pre-trained ResNet18."""
    model = create_resnet18(pretrained=True, num_classes=len(class_names))
    model = apply_peft_or_manual_adapter(model)

    trainable, total = count_trainable_parameters(model)
    print(f"PEFT/adapter trainable params: {trainable:,} / {total:,}")

    optimizer = optim.AdamW(
        filter(lambda parameter: parameter.requires_grad, model.parameters()),
        lr=5e-4,
        weight_decay=1e-4,
    )

    model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        num_epochs=num_epochs,
        artifact_path=ARTIFACT_DIR / "resnet18_peft_adapter.pth",
        log_path=ARTIFACT_DIR / "resnet18_peft_adapter_log.json",
        freeze_batchnorm=True,
    )
    return model, history

# %% [markdown]
# ---
# # Section 6 — Training Loop with Class Imbalance Handling
#
# The target dataset is highly imbalanced, approximately 80% normal and
# 20% defective.
#
# To compensate, we use weighted cross-entropy loss:
#
# ```python
# criterion = nn.CrossEntropyLoss(weight=class_weights)
# ```
#
# The minority class receives a larger loss weight.
#
# ## Data Augmentation
#
# Training transforms are intentionally stronger:
#
# - random resized crops
# - horizontal / vertical flips
# - small rotations
# - color jitter
#
# Validation transforms are deterministic:
#
# - resize
# - normalize
#
# ## CPU Runtime Trade-off
#
# For the 4-way comparison, this notebook uses few epochs and may cap the
# dataset size. This is done explicitly to keep CPU execution feasible.

# %%
train_transform = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(
            brightness=0.15,
            contrast=0.15,
            saturation=0.10,
            hue=0.02,
        ),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

train_dataset_full = datasets.ImageFolder(
    root=DATA_ROOT / "train",
    transform=train_transform,
)

val_dataset_full = datasets.ImageFolder(
    root=DATA_ROOT / "val",
    transform=val_transform,
)

class_names = train_dataset_full.classes
print(f"Classes: {class_names}")

# %%
def make_subset(
    dataset: datasets.ImageFolder,
    max_samples: Optional[int],
    seed: int = SEED,
) -> datasets.ImageFolder | Subset:
    """Return a deterministic subset if max_samples is set."""
    if max_samples is None or max_samples >= len(dataset):
        return dataset

    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(dataset), generator=generator)[:max_samples]
    return Subset(dataset, indices.tolist())


train_dataset = make_subset(train_dataset_full, MAX_TRAIN_SAMPLES)
val_dataset = make_subset(val_dataset_full, MAX_VAL_SAMPLES)

print(f"Training samples used: {len(train_dataset)}")
print(f"Validation samples used: {len(val_dataset)}")

# %%
def get_targets(dataset: datasets.ImageFolder | Subset) -> List[int]:
    """Extract class targets from ImageFolder or Subset."""
    if isinstance(dataset, Subset):
        return [dataset.dataset.targets[index] for index in dataset.indices]
    return list(dataset.targets)


def compute_class_weights(
    targets: List[int],
    num_classes: int,
    device: torch.device,
) -> torch.Tensor:
    """Compute inverse-frequency class weights safely."""
    counts = np.bincount(targets, minlength=num_classes)
    print(f"Class counts: {dict(zip(class_names, counts))}")

    if np.any(counts == 0):
        print(
            "Warning: At least one class has zero samples. "
            "Training/evaluation metrics may be unreliable."
        )

    safe_counts = np.maximum(counts, 1)
    total = safe_counts.sum()
    weights = total / (num_classes * safe_counts)
    return torch.tensor(weights, dtype=torch.float32, device=device)


train_targets = get_targets(train_dataset)
class_weights = compute_class_weights(
    train_targets,
    num_classes=len(class_names),
    device=DEVICE,
)

print(f"Class weights: {class_weights}")

criterion = nn.CrossEntropyLoss(weight=class_weights)

# %%
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False,
)

# %%
def set_batchnorm_eval(model: nn.Module) -> None:
    """Keep BatchNorm layers in eval mode for stable small-data fine-tuning."""
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()


def run_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: Optional[optim.Optimizer],
    device: torch.device,
    train: bool,
    freeze_batchnorm: bool = False,
) -> Tuple[float, float]:
    """Run one training or validation epoch."""
    if train:
        model.train()
        if freeze_batchnorm:
            set_batchnorm_eval(model)
    else:
        model.eval()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if train and optimizer is not None:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            outputs = model(images)
            loss = criterion(outputs, labels)
            predictions = outputs.argmax(dim=1)

            if train and optimizer is not None:
                loss.backward()
                optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (predictions == labels).sum().item()
        running_total += batch_size

    epoch_loss = running_loss / max(running_total, 1)
    epoch_accuracy = running_correct / max(running_total, 1)

    return epoch_loss, epoch_accuracy


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
    num_epochs: int,
    artifact_path: Path,
    log_path: Path,
    freeze_batchnorm: bool = False,
) -> Tuple[nn.Module, List[Dict[str, float]]]:
    """Reusable training loop with per-epoch loss/accuracy logging."""
    model = model.to(device)

    best_model_state = copy.deepcopy(model.state_dict())
    best_val_accuracy = -math.inf
    history = []

    for epoch in range(num_epochs):
        train_loss, train_accuracy = run_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            train=True,
            freeze_batchnorm=freeze_batchnorm,
        )

        val_loss, val_accuracy = run_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            optimizer=None,
            device=device,
            train=False,
            freeze_batchnorm=False,
        )

        epoch_record = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
        }
        history.append(epoch_record)

        print(
            f"Epoch {epoch + 1:02d}/{num_epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"train_acc={train_accuracy:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_accuracy:.4f}"
        )

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_state)

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
            "history": history,
        },
        artifact_path,
    )

    with open(log_path, "w", encoding="utf-8") as file:
        json.dump(history, file, indent=2)

    print(f"Saved model artifact to: {artifact_path}")
    print(f"Saved training log to: {log_path}")

    return model, history

# %% [markdown]
# ## Orchestrating Cell
#
# The three transfer-learning functions from Section 5 are called sequentially
# here, after the shared dataset, weighted loss, and reusable training loop
# have been defined.

# %%
transfer_results = {}

print("\n=== Training full-freeze model ===")
full_freeze_model, full_freeze_history = train_full_freeze(
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    class_names=class_names,
    device=DEVICE,
)

transfer_results["full_freeze"] = {
    "model": full_freeze_model,
    "history": full_freeze_history,
}

print("\n=== Training layer4-unfreeze model ===")
layer4_model, layer4_history = train_unfreeze_layer4(
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    class_names=class_names,
    device=DEVICE,
)

transfer_results["layer4_unfreeze"] = {
    "model": layer4_model,
    "history": layer4_history,
}

print("\n=== Training PEFT / adapter model ===")
peft_model, peft_history = train_peft(
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    class_names=class_names,
    device=DEVICE,
)

transfer_results["peft_adapter"] = {
    "model": peft_model,
    "history": peft_history,
}

# %% [markdown]
# ---
# # Section 7 — Baseline, Evaluation & QA
#
# We now train a from-scratch ResNet18 as a control baseline.
#
# This model:
#
# - Uses random initialization
# - Has no ImageNet pre-training
# - Is fully trainable
#
# On a very small dataset, this baseline often overfits or underperforms
# transfer-learning variants. However, it is important for detecting
# **negative transfer**.
#
# Negative transfer occurs when pre-trained features hurt performance compared
# with training from scratch.

# %%
def train_from_scratch_baseline(
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    class_names: List[str],
    device: torch.device,
    num_epochs: int = NUM_EPOCHS_BASELINE,
) -> Tuple[nn.Module, List[Dict[str, float]]]:
    """Train a randomly initialized ResNet18 baseline."""
    model = create_resnet18(pretrained=False, num_classes=len(class_names))

    trainable, total = count_trainable_parameters(model)
    print(f"From-scratch trainable params: {trainable:,} / {total:,}")

    optimizer = optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4,
    )

    model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        num_epochs=num_epochs,
        artifact_path=ARTIFACT_DIR / "resnet18_from_scratch.pth",
        log_path=ARTIFACT_DIR / "resnet18_from_scratch_log.json",
        freeze_batchnorm=False,
    )

    return model, history


print("\n=== Training from-scratch baseline ===")
baseline_model, baseline_history = train_from_scratch_baseline(
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    class_names=class_names,
    device=DEVICE,
)

transfer_results["from_scratch"] = {
    "model": baseline_model,
    "history": baseline_history,
}

# %% [markdown]
# ## Evaluation Utilities
#
# We compute:
#
# - Accuracy
# - Confusion matrix
# - Per-class precision
# - Per-class recall
# - Overfitting gap
#
# The overfitting gap is:
#
# ```text
# final_train_accuracy - final_validation_accuracy
# ```
#
# Metrics use `zero_division=0` to gracefully handle edge cases where a class
# receives no predictions.

# %%
def predict_all(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray]:
    """Collect all predictions and labels from a data loader."""
    model = model.to(device)
    model.eval()

    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            predictions = outputs.argmax(dim=1).cpu().numpy()

            all_predictions.extend(predictions)
            all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_predictions)


def evaluate_model(
    name: str,
    model: nn.Module,
    history: List[Dict[str, float]],
    loader: DataLoader,
    class_names: List[str],
    device: torch.device,
) -> Dict[str, float]:
    """Evaluate a model and return flat metric records."""
    labels, predictions = predict_all(model, loader, device)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, _, _ = precision_recall_fscore_support(
        labels,
        predictions,
        labels=list(range(len(class_names))),
        zero_division=0,
    )

    final_train_accuracy = history[-1]["train_accuracy"]
    final_val_accuracy = history[-1]["val_accuracy"]
    overfitting_gap = final_train_accuracy - final_val_accuracy

    record = {
        "variant": name,
        "accuracy": accuracy,
        "overfitting_gap": overfitting_gap,
    }

    for idx, class_name in enumerate(class_names):
        record[f"precision_{class_name}"] = precision[idx]
        record[f"recall_{class_name}"] = recall[idx]

    return record


evaluation_records = []

for variant_name, result in transfer_results.items():
    record = evaluate_model(
        name=variant_name,
        model=result["model"],
        history=result["history"],
        loader=val_loader,
        class_names=class_names,
        device=DEVICE,
    )
    evaluation_records.append(record)

comparison_df = pd.DataFrame(evaluation_records)
comparison_df

# %% [markdown]
# ## Confusion Matrices

# %%
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for axis, (variant_name, result) in zip(axes, transfer_results.items()):
    labels, predictions = predict_all(result["model"], val_loader, DEVICE)
    cm = confusion_matrix(
        labels,
        predictions,
        labels=list(range(len(class_names))),
    )

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names,
    )
    display.plot(ax=axis, colorbar=False)
    axis.set_title(variant_name)

plt.tight_layout()
plt.show()

# %% [markdown]
# ## Final Comparison Plot

# %%
plot_df = comparison_df.set_index("variant")

metrics_to_plot = ["accuracy", "overfitting_gap"]
plot_df[metrics_to_plot].plot(kind="bar", figsize=(10, 5))
plt.title("Model Comparison: Accuracy and Overfitting Gap")
plt.ylabel("Score")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# %%
precision_recall_columns = [
    column
    for column in comparison_df.columns
    if column.startswith("precision_") or column.startswith("recall_")
]

comparison_df.set_index("variant")[precision_recall_columns].plot(
    kind="bar",
    figsize=(12, 5),
)
plt.title("Per-Class Precision and Recall by Variant")
plt.ylabel("Score")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# %% [markdown]
# ## Sanity Check: Negative Transfer
#
# Compare the transfer-learning variants against the from-scratch baseline.
#
# If none of the transfer variants outperform the baseline, possible reasons
# include:
#
# - The target domain is too different from ImageNet.
# - The labels are noisy or visually ambiguous.
# - The dataset is too small for reliable validation metrics.
# - The class imbalance is severe.
# - The number of epochs was intentionally reduced for CPU runtime.
# - The fallback Hymenoptera redistribution creates artificial labels, so the
#   task may not reflect meaningful visual differences.
#
# In a real manufacturing dataset, transfer learning should be evaluated using
# a stable validation or test set and domain-specific inspection.

# %%
baseline_accuracy = comparison_df.loc[
    comparison_df["variant"] == "from_scratch",
    "accuracy",
].iloc[0]

transfer_only = comparison_df[
    comparison_df["variant"].isin(
        ["full_freeze", "layer4_unfreeze", "peft_adapter"]
    )
]

best_transfer_accuracy = transfer_only["accuracy"].max()

if best_transfer_accuracy > baseline_accuracy:
    print("At least one transfer variant outperformed the from-scratch baseline.")
else:
    print("No transfer variant outperformed the from-scratch baseline.")
    print("This may indicate negative transfer or CPU-limited undertraining.")

comparison_df.sort_values("accuracy", ascending=False)

# %% [markdown]
# ---
# # Section 8 — Reflection: When Would Transfer Fail?
#
# Transfer learning depends on the **relatedness assumption**:
#
# > Features learned on the source task should be useful for the target task.
#
# ImageNet features often transfer well to ordinary 2D photographic image
# tasks because many datasets share visual primitives such as edges, textures,
# lighting gradients, and object contours.
#
# However, transfer may fail when the target data is structurally unlike
# ImageNet, for example:
#
# - RF spectrograms
# - medical volumetric scans
# - microscopy volumes
# - satellite radar imagery
# - thermal sensor maps
# - non-visual tabular data encoded as images
#
# In those cases, ImageNet filters may encode irrelevant biases. This can lead
# to **negative transfer**, where a pre-trained model performs worse than a
# carefully trained baseline.
#
# ## Reader Exercise
#
# Consider a target task where the input is an RF spectrogram.
#
# Ask:
#
# - Do ImageNet edges and textures correspond to meaningful RF patterns?
# - Are late ImageNet object features useful?
# - Would full freezing help or hurt?
# - Would adapter-based tuning be safer than full fine-tuning?
# - Would self-supervised pre-training on unlabeled target-domain data be
#   more appropriate?
#
# The answer to these questions should drive your freezing and fine-tuning
# strategy.

# %% [markdown]
# ---
# # Final QA Pass
#
# ## 1. API Check
#
# ✅ Uses the current torchvision weights API:
#
# ```python
# models.resnet18(weights=ResNet18_Weights.DEFAULT)
# models.resnet18(weights=None)
# ```
#
# ❌ Does not use deprecated:
#
# ```python
# pretrained=True
# ```
#
# ## 2. Edge Cases
#
# ✅ CPU-only execution supported through automatic device detection.
#
# ✅ Missing `peft` library is handled with a manual adapter fallback.
#
# ✅ If PEFT fails on Conv2d modules at runtime, the notebook falls back to
# manual adapters.
#
# ✅ Weighted loss handles class imbalance.
#
# ✅ Class-weight computation avoids division by zero if a class is missing.
#
# ✅ Precision/recall use:
#
# ```python
# zero_division=0
# ```
#
# ✅ Training loop is safe for batches containing only one class.
#
# ## 3. Hallucination Check
#
# The PEFT integration is wrapped in a runtime `try` / `except` because PEFT
# support for torchvision convolutional modules can vary by version.
#
# The notebook does not silently assume that PEFT will work. If PEFT fails,
# the manual adapter implementation is used instead.